In [ ]:
!pip install -q chefshatgym --no-deps
!pip install -q nest_asyncio torch numpy matplotlib pandas websockets

In [ ]:
import asyncio
import random
import os
import math
import collections
import numpy as np

if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import nest_asyncio
nest_asyncio.apply()

from ChefsHatGym.agents.base_classes.chefs_hat_player import ChefsHatPlayer
from ChefsHatGym.gameRooms.chefs_hat_room_local import ChefsHatRoomLocal
from ChefsHatGym.agents.agent_random import AgentRandon as RandomAgent

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


Device : cpu


In [13]:
STUDENT_ID = 16533318
VARIANT    = STUDENT_ID % 7

VARIANT_MAP = {
    0: 'Opponent Modelling',
    1: 'Opponent Modelling',
    2: 'Sparse / Delayed Reward',
    3: 'Sparse / Delayed Reward',
    4: 'Partial Observability',
    5: 'Robustness and Generalisation',
    6: 'Generative AI Augmentation',
}

print(f'Student ID : {STUDENT_ID}')
print(f'Variant    : {VARIANT} — {VARIANT_MAP[VARIANT]}')
print()
print('Approach:')
print('  Baseline  : MLP-DQN on full 228-dim observation (board + hand + action_mask)')
print('  Partial   : LSTM-DQN on 217-dim observation (hand + action_mask; board hidden)')
print('  Rationale : The LSTM accumulates temporal context to recover information')
print('              that is absent from each individual partial observation.')

Student ID : 16533318
Variant    : 4 — Partial Observability

Approach:
  Baseline  : MLP-DQN on full 228-dim observation (board + hand + action_mask)
  Partial   : LSTM-DQN on 217-dim observation (hand + action_mask; board hidden)
  Rationale : The LSTM accumulates temporal context to recover information
              that is absent from each individual partial observation.


In [14]:
# Observation layout (ChefsHatGym v3):
#   indices  0 -  10 : board / pizza area   (11 slots)
#   indices 11 -  27 : player hand cards    (17 slots, 0 = empty)
#   indices 28 - 227 : valid action mask    (200 slots, 1 = legal)
# Full obs  : 228 dims   (all three)
# Partial   : 217 dims   (hand + mask only; board omitted)

OBS_FULL_DIM    = 228
OBS_PARTIAL_DIM = 217
N_ACTIONS       = 200
HAND_DIM        = 17
BOARD_DIM       = 11


def extract_partial(observation):
    """Drop board slice; return hand + action_mask (217 dims)."""
    obs = np.asarray(observation, dtype=np.float32)
    return obs[BOARD_DIM:]   # indices 11..227


def action_mask_from_obs(observation):
    """Return the 200-element valid-action binary vector."""
    obs = np.asarray(observation, dtype=np.float32)
    return obs[28:]

In [15]:
Transition = collections.namedtuple(
    'Transition', ['obs', 'action', 'reward', 'next_obs', 'done']
)


class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self._buf = deque(maxlen=capacity)

    def push(self, obs, action, reward, next_obs, done):
        self._buf.append(Transition(obs, action, reward, next_obs, done))

    def sample(self, n):
        return random.sample(self._buf, n)

    def __len__(self):
        return len(self._buf)


class EpisodeBuffer:
    """Stores complete episodes for sequence-based LSTM training."""

    def __init__(self, capacity=2_000):
        self._buf     = deque(maxlen=capacity)
        self._current = []

    def start_episode(self):
        self._current = []

    def push_step(self, obs, action, reward, next_obs, done):
        self._current.append((obs, action, reward, next_obs, done))

    def end_episode(self):
        if self._current:
            self._buf.append(list(self._current))
        self._current = []

    def sample_sequences(self, n_eps, seq_len=8):
        episodes = random.sample(self._buf, min(n_eps, len(self._buf)))
        seqs = []
        for ep in episodes:
            if len(ep) <= seq_len:
                pad = [(ep[0][0], 0, 0.0, ep[0][3], True)] * (seq_len - len(ep))
                seq = pad + ep
            else:
                start = random.randint(0, len(ep) - seq_len)
                seq   = ep[start:start + seq_len]
            seqs.append(seq)
        return seqs

    def __len__(self):
        return len(self._buf)

In [16]:
class MLPDQN(nn.Module):
    """Feed-forward DQN for full observations."""

    def __init__(self, obs_dim=OBS_FULL_DIM, n_actions=N_ACTIONS, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


class LSTMDQNNet(nn.Module):
    """Recurrent DQN for partial observations.

    Embedding -> LSTM -> Q-head.
    The LSTM hidden state persists across a match to accumulate
    temporal context replacing the missing board slice.
    """

    def __init__(self, obs_dim=OBS_PARTIAL_DIM, n_actions=N_ACTIONS,
                 embed_dim=128, lstm_hidden=128, n_layers=1):
        super().__init__()
        self.embed_dim   = embed_dim
        self.lstm_hidden = lstm_hidden
        self.n_layers    = n_layers

        self.embed = nn.Sequential(
            nn.Linear(obs_dim, embed_dim),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(
            input_size  = embed_dim,
            hidden_size = lstm_hidden,
            num_layers  = n_layers,
            batch_first = True,
        )
        self.head = nn.Linear(lstm_hidden, n_actions)

    def forward(self, x, hidden=None):
        # x: (B, T, obs_dim) or (B, obs_dim) for single-step inference
        single_step = (x.dim() == 2)
        if single_step:
            x = x.unsqueeze(1)
        e = self.embed(x)
        out, hidden_out = self.lstm(e, hidden)
        q = self.head(out)
        if single_step:
            q = q.squeeze(1)
        return q, hidden_out

    def init_hidden(self, batch_size=1):
        h = torch.zeros(self.n_layers, batch_size, self.lstm_hidden, device=DEVICE)
        c = torch.zeros(self.n_layers, batch_size, self.lstm_hidden, device=DEVICE)
        return (h, c)

In [17]:
def eps_threshold(steps_done, eps_start, eps_end, eps_decay):
    return eps_end + (eps_start - eps_end) * math.exp(-steps_done / eps_decay)


def masked_argmax(q_np, action_mask_np):
    q = q_np.copy()
    q[action_mask_np == 0] = -1e9
    return int(np.argmax(q))


def random_valid_action(action_mask_np):
    valid = np.where(action_mask_np > 0)[0]
    return int(np.random.choice(valid)) if len(valid) > 0 else 0


def hot_encode(action_idx, n=N_ACTIONS):
    v = [0] * n
    v[action_idx] = 1
    return v


def extract_score(final_scores, agent_name):
    """Safely extract performance metrics from room.final_scores."""
    if final_scores is None:
        return 0.0, 0
    entry = final_scores.get(agent_name, {})
    perf  = float(entry.get('performance_score', entry.get('performanceScore', 0.0)))
    score = int(entry.get('score', 0))
    return perf, score


def rolling_mean(data, window=10):
    return pd.Series(data).rolling(window, min_periods=1).mean().tolist()

In [18]:
class FullObsDQNAgent(ChefsHatPlayer):
    """DQN agent trained on the complete 228-dim observation."""

    def __init__(self, name, log_directory='',
                 lr=1e-3, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay=500,
                 batch_size=64, target_update=10,
                 buffer_capacity=10_000):
        super().__init__('FullObsDQN', name, log_directory=log_directory)

        self.gamma        = gamma
        self.batch_size   = batch_size
        self.eps_start    = eps_start
        self.eps_end      = eps_end
        self.eps_decay    = eps_decay
        self.target_update = target_update

        self.policy_net   = MLPDQN().to(DEVICE)
        self.target_net   = MLPDQN().to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer    = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer       = ReplayBuffer(buffer_capacity)

        self.steps_done   = 0
        self.updates_done = 0
        self.losses       = []
        self.match_rewards = []

        self._prev_obs = None
        self._prev_act = None

    def update_start_match(self, cards, players, starting_player):
        self._prev_obs = None
        self._prev_act = None

    def get_action(self, observation):
        obs  = np.asarray(observation, dtype=np.float32)
        mask = action_mask_from_obs(obs)
        eps  = eps_threshold(self.steps_done, self.eps_start, self.eps_end, self.eps_decay)
        self.steps_done += 1

        if random.random() < eps:
            act = random_valid_action(mask)
        else:
            with torch.no_grad():
                q = self.policy_net(
                    torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
                ).squeeze(0).cpu().numpy()
            act = masked_argmax(q, mask)

        if self._prev_obs is not None:
            self.buffer.push(self._prev_obs, self._prev_act, 0.0, obs, False)

        self._prev_obs = obs
        self._prev_act = act
        self._maybe_learn()
        return hot_encode(act)

    def get_exhanged_cards(self, cards, amount):
        return sorted(cards)[:amount]

    def do_special_action(self, info, special_action):
        return True

    def update_my_action(self, env_info):
        pass

    def update_action_others(self, env_info):
        pass

    def update_exchange_cards(self, cards, amount):
        pass

    def observe_special_action(self, action, player):
        pass

    def update_end_match(self, env_info):
        reward = self.get_reward(env_info)
        self.match_rewards.append(reward)
        if self._prev_obs is not None:
            self.buffer.push(self._prev_obs, self._prev_act, reward, self._prev_obs, True)
        self._prev_obs = None
        self._prev_act = None

    def get_reward(self, env_info):
        scores = env_info.get('Match_Score', env_info.get('match_score', []))
        idx    = env_info.get('Author_Index', env_info.get('author_index', 0))
        if scores and idx < len(scores):
            return float(scores[idx])
        return 0.0

    def update_game_over(self):
        pass

    def _maybe_learn(self):
        if len(self.buffer) < self.batch_size:
            return
        batch  = self.buffer.sample(self.batch_size)
        obs_t  = torch.FloatTensor(np.stack([t.obs      for t in batch])).to(DEVICE)
        act_t  = torch.LongTensor([t.action   for t in batch]).to(DEVICE)
        rew_t  = torch.FloatTensor([t.reward   for t in batch]).to(DEVICE)
        nobs_t = torch.FloatTensor(np.stack([t.next_obs for t in batch])).to(DEVICE)
        done_t = torch.FloatTensor([float(t.done) for t in batch]).to(DEVICE)

        q_vals = self.policy_net(obs_t).gather(1, act_t.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q = self.target_net(nobs_t).max(1)[0]
        targets = rew_t + self.gamma * next_q * (1.0 - done_t)

        loss = F.smooth_l1_loss(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        self.losses.append(loss.item())
        self.updates_done += 1
        if self.updates_done % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())


In [19]:
class LSTMDQNAgent(ChefsHatPlayer):
    """LSTM-DQN operating on partial observations (board state withheld).

    The LSTM hidden state carries temporal context across a match,
    allowing the agent to implicitly track board dynamics from history.
    """

    def __init__(self, name, log_directory='',
                 lr=1e-3, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay=500,
                 batch_size=8, target_update=10,
                 buffer_capacity=2_000,
                 lstm_hidden=128, seq_len=8):
        super().__init__('LSTMDQNAgent', name, log_directory=log_directory)

        self.gamma        = gamma
        self.batch_size   = batch_size
        self.eps_start    = eps_start
        self.eps_end      = eps_end
        self.eps_decay    = eps_decay
        self.target_update = target_update
        self.seq_len      = seq_len

        self.policy_net   = LSTMDQNNet(lstm_hidden=lstm_hidden).to(DEVICE)
        self.target_net   = LSTMDQNNet(lstm_hidden=lstm_hidden).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer    = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.ep_buffer    = EpisodeBuffer(buffer_capacity)

        self.steps_done   = 0
        self.updates_done = 0
        self.losses       = []
        self.match_rewards = []

        self._hidden   = None
        self._prev_obs = None
        self._prev_act = None

    def update_start_match(self, cards, players, starting_player):
        self._hidden   = self.policy_net.init_hidden(batch_size=1)
        self._prev_obs = None
        self._prev_act = None
        self.ep_buffer.start_episode()

    def get_action(self, observation):
        obs_full   = np.asarray(observation, dtype=np.float32)
        obs_part   = extract_partial(obs_full)          # 217 dims (no board)
        mask       = action_mask_from_obs(obs_full)     # 200 dims

        eps = eps_threshold(self.steps_done, self.eps_start, self.eps_end, self.eps_decay)
        self.steps_done += 1

        x_t = torch.FloatTensor(obs_part).unsqueeze(0).to(DEVICE)

        if random.random() < eps:
            act = random_valid_action(mask)
            with torch.no_grad():
                _, self._hidden = self.policy_net(x_t, self._hidden)
        else:
            self.policy_net.eval()
            with torch.no_grad():
                q_t, self._hidden = self.policy_net(x_t, self._hidden)
            self.policy_net.train()
            act = masked_argmax(q_t.squeeze(0).cpu().numpy(), mask)

        if self._prev_obs is not None:
            self.ep_buffer.push_step(self._prev_obs, self._prev_act, 0.0, obs_part, False)

        self._prev_obs = obs_part
        self._prev_act = act
        return hot_encode(act)

    def get_exhanged_cards(self, cards, amount):
        return sorted(cards)[:amount]

    def do_special_action(self, info, special_action):
        return True

    def update_my_action(self, env_info):
        pass

    def update_action_others(self, env_info):
        pass

    def update_exchange_cards(self, cards, amount):
        pass

    def observe_special_action(self, action, player):
        pass

    def update_end_match(self, env_info):
        reward = self.get_reward(env_info)
        self.match_rewards.append(reward)
        if self._prev_obs is not None:
            self.ep_buffer.push_step(self._prev_obs, self._prev_act,
                                     reward, self._prev_obs, True)
        self.ep_buffer.end_episode()
        self._maybe_learn()

    def get_reward(self, env_info):
        scores = env_info.get('Match_Score', env_info.get('match_score', []))
        idx    = env_info.get('Author_Index', env_info.get('author_index', 0))
        if scores and idx < len(scores):
            return float(scores[idx])
        return 0.0

    def update_game_over(self):
        pass

    def _maybe_learn(self):
        if len(self.ep_buffer) < self.batch_size:
            return
        seqs = self.ep_buffer.sample_sequences(self.batch_size, self.seq_len)

        obs_t  = torch.FloatTensor(np.array([[s[0] for s in seq] for seq in seqs])).to(DEVICE)
        act_t  = torch.LongTensor([[s[1] for s in seq] for seq in seqs]).to(DEVICE)
        rew_t  = torch.FloatTensor([[s[2] for s in seq] for seq in seqs]).to(DEVICE)
        nobs_t = torch.FloatTensor(np.array([[s[3] for s in seq] for seq in seqs])).to(DEVICE)
        done_t = torch.FloatTensor([[float(s[4]) for s in seq] for seq in seqs]).to(DEVICE)

        q_all, _  = self.policy_net(obs_t)
        q_vals    = q_all.gather(2, act_t.unsqueeze(2)).squeeze(2)
        with torch.no_grad():
            nq_all, _ = self.target_net(nobs_t)
            next_q    = nq_all.max(2)[0]
        targets = rew_t + self.gamma * next_q * (1.0 - done_t)

        loss = F.smooth_l1_loss(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        self.losses.append(loss.item())
        self.updates_done += 1
        if self.updates_done % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())


In [20]:
def run_game(agent, n_matches, log_dir, game_name):
    """Run a local ChefsHat game with n_matches stops against 3 random opponents."""
    room = ChefsHatRoomLocal(
        room_name            = game_name,
        game_type            = 'MATCHES',
        stop_criteria        = n_matches,
        log_directory        = log_dir,
        verbose_console      = False,
        verbose_log          = False,
        game_verbose_console = False,
        game_verbose_log     = False,
    )
    opponents = [
        RandomAgent(name=f'Rand{i}', log_directory=log_dir)
        for i in range(3)
    ]
    for p in [agent] + opponents:
        room.add_player(p)
    info = room.start_new_game()
    return info


def train_agent(agent, n_games=50, n_matches=10, log_dir='./logs', tag=''):
    """Run n_games of ChefsHat, each with n_matches, collecting metrics."""
    os.makedirs(log_dir, exist_ok=True)

    perf_scores  = []
    win_flags    = []
    loss_log     = []

    for g in range(n_games):
        game_name = f'{tag}_g{g}'
        n_before  = len(agent.match_rewards)

        run_game(agent, n_matches, log_dir, game_name)

        # Collect rewards earned during this game
        game_rewards = agent.match_rewards[n_before:]
        perf = float(np.mean(game_rewards)) if game_rewards else 0.0
        win  = 1 if (game_rewards and game_rewards[-1] > 0) else 0

        perf_scores.append(perf)
        win_flags.append(win)

        recent_loss = float(np.nanmean(agent.losses[-50:])) if agent.losses else float('nan')
        loss_log.append(recent_loss)

        if (g + 1) % 10 == 0:
            eps = eps_threshold(agent.steps_done, agent.eps_start,
                                agent.eps_end, agent.eps_decay)
            print(f'[{tag}] game {g+1:3d}/{n_games} | '
                  f'perf={perf:.3f} | wr={np.mean(win_flags):.2f} | '
                  f'loss={recent_loss:.4f} | eps={eps:.3f}')

    return {
        'perf_scores' : perf_scores,
        'win_flags'   : win_flags,
        'loss_log'    : loss_log,
        'raw_losses'  : list(agent.losses),
    }


In [ ]:
N_GAMES    = 60
N_MATCHES  = 10

print('Training Full-Observation MLP-DQN ...')
full_agent = FullObsDQNAgent(
    name='FullObsDQN',
    log_directory='./logs_full',
    lr=1e-3, gamma=0.99,
    eps_start=1.0, eps_end=0.05, eps_decay=400,
    batch_size=64, target_update=10,
)
full_res = train_agent(full_agent, n_games=N_GAMES, n_matches=N_MATCHES,
                       log_dir='./logs_full', tag='FULL')
print(f'Done. Updates: {full_agent.updates_done}')

Training Full-Observation MLP-DQN ...
Agent FullObsDQN_FullObsDQN log folder:  c:\Users\sayan\Downloads\Task Copy 2\logs_full\FullObsDQN_FullObsDQN
Agent RANDOM_Rand0 log folder:  c:\Users\sayan\Downloads\Task Copy 2\logs_full\RANDOM_Rand0
Agent RANDOM_Rand1 log folder:  c:\Users\sayan\Downloads\Task Copy 2\logs_full\RANDOM_Rand1
Agent RANDOM_Rand2 log folder:  c:\Users\sayan\Downloads\Task Copy 2\logs_full\RANDOM_Rand2
Log Name: 1772381534.379824
Log Directory: ./logs_full\16-1214_FULL_g0\Log\Log.log
Verbose: False
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 

[Room]: [Room][ERROR]: ---- Invalid action!


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 1.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 

In [ ]:
print('Training Partial-Observation LSTM-DQN ...')
lstm_agent = LSTMDQNAgent(
    name='LSTMPartialDQN',
    log_directory='./logs_lstm',
    lr=1e-3, gamma=0.99,
    eps_start=1.0, eps_end=0.05, eps_decay=400,
    batch_size=8, target_update=10,
    lstm_hidden=128, seq_len=8,
)
lstm_res = train_agent(lstm_agent, n_games=N_GAMES, n_matches=N_MATCHES,
                       log_dir='./logs_lstm', tag='LSTM')
print(f'Done. Updates: {lstm_agent.updates_done}')

In [ ]:
summary = pd.DataFrame([
    {
        'Agent'                 : 'Full-Obs MLP-DQN',
        'Avg Perf Score'        : round(np.mean(full_res['perf_scores']),  4),
        'Win Rate'              : round(np.mean(full_res['win_flags']),     4),
        'Final 10 Win Rate'     : round(np.mean(full_res['win_flags'][-10:]), 4),
        'Avg Loss'              : round(np.nanmean(full_res['loss_log']),   4),
        'Network Updates'       : full_agent.updates_done,
        'Trainable Params'      : sum(p.numel() for p in full_agent.policy_net.parameters()),
    },
    {
        'Agent'                 : 'Partial LSTM-DQN',
        'Avg Perf Score'        : round(np.mean(lstm_res['perf_scores']),  4),
        'Win Rate'              : round(np.mean(lstm_res['win_flags']),     4),
        'Final 10 Win Rate'     : round(np.mean(lstm_res['win_flags'][-10:]), 4),
        'Avg Loss'              : round(np.nanmean(lstm_res['loss_log']),   4),
        'Network Updates'       : lstm_agent.updates_done,
        'Trainable Params'      : sum(p.numel() for p in lstm_agent.policy_net.parameters()),
    },
])
print(summary.to_string(index=False))

In [ ]:
x = np.arange(1, N_GAMES + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for label, res, c in [
    ('Full-Obs DQN',     full_res, 'steelblue'),
    ('Partial LSTM-DQN', lstm_res, 'darkorange'),
]:
    axes[0].plot(x, rolling_mean(res['perf_scores']), label=label, color=c)
    axes[1].plot(x, rolling_mean(res['win_flags']),   label=label, color=c)

for ax, title, ylabel in zip(
    axes[:2],
    ['Performance Score (rolling-10)', 'Win Rate (rolling-10)'],
    ['Score', 'Win Rate'],
):
    ax.set_title(title)
    ax.set_xlabel('Game')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.3)

for label, res, c in [
    ('Full-Obs DQN',     full_res, 'steelblue'),
    ('Partial LSTM-DQN', lstm_res, 'darkorange'),
]:
    rl = res['raw_losses']
    if rl:
        lx = np.linspace(1, N_GAMES, len(rl))
        sm = pd.Series(rl).rolling(50, min_periods=1).mean()
        axes[2].plot(lx, sm, label=label, color=c)

axes[2].set_title('Training Loss (rolling-50 steps)')
axes[2].set_xlabel('Update step')
axes[2].set_ylabel('Huber Loss')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: learning_curves.png')

In [ ]:
# Stability: rolling mean +/- std band
fig, ax = plt.subplots(figsize=(10, 5))

for label, res, c in [
    ('Full-Obs DQN',     full_res, 'steelblue'),
    ('Partial LSTM-DQN', lstm_res, 'darkorange'),
]:
    s   = pd.Series(res['perf_scores'])
    rm  = s.rolling(10, min_periods=1).mean().values
    std = s.rolling(10, min_periods=1).std().fillna(0).values
    ax.plot(x, rm, label=label, color=c)
    ax.fill_between(x, rm - std, rm + std, alpha=0.2, color=c)

ax.set_title('Performance Score: rolling mean +/- std (window=10)')
ax.set_xlabel('Game')
ax.set_ylabel('Performance Score')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('stability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stability.png')

In [ ]:
# Hyperparameter experiment: LSTM hidden size
HIDDEN_SIZES = [64, 128, 256]
HP_GAMES     = 30
hp_results   = {}

for hs in HIDDEN_SIZES:
    print(f'  hidden_size={hs} ...')
    ag = LSTMDQNAgent(
        name=f'LSTM_h{hs}', log_directory=f'./logs_hp{hs}',
        lr=1e-3, eps_start=1.0, eps_end=0.05, eps_decay=300,
        gamma=0.99, lstm_hidden=hs,
    )
    hp_results[hs] = train_agent(
        ag, n_games=HP_GAMES, n_matches=N_MATCHES,
        log_dir=f'./logs_hp{hs}', tag=f'hp{hs}'
    )

hp_table = pd.DataFrame([
    {
        'LSTM hidden'      : hs,
        'Mean Perf Score'  : round(np.mean(r['perf_scores']), 4),
        'Mean Win Rate'    : round(np.mean(r['win_flags']),   4),
        'Final-10 Win Rate': round(np.mean(r['win_flags'][-10:]), 4),
    }
    for hs, r in hp_results.items()
])
print(hp_table.to_string(index=False))

xhp = np.arange(1, HP_GAMES + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for hs, r in hp_results.items():
    axes[0].plot(xhp, rolling_mean(r['perf_scores']), label=f'h={hs}')
    axes[1].plot(xhp, rolling_mean(r['win_flags']),   label=f'h={hs}')
for ax, title, yl in zip(axes,
    ['LSTM Hidden Size vs Performance', 'LSTM Hidden Size vs Win Rate'],
    ['Performance Score', 'Win Rate']):
    ax.set_title(title)
    ax.set_xlabel('Game')
    ax.set_ylabel(yl)
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('hp_hidden_size.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hp_hidden_size.png')

In [ ]:
# Exploration strategy experiment: epsilon decay rate
DECAY_VALS = [200, 400, 800]
EPS_GAMES  = 30
eps_results = {}

for decay in DECAY_VALS:
    print(f'  eps_decay={decay} ...')
    ag = LSTMDQNAgent(
        name=f'LSTM_d{decay}', log_directory=f'./logs_eps{decay}',
        lr=1e-3, eps_start=1.0, eps_end=0.05, eps_decay=decay,
        gamma=0.99, lstm_hidden=128,
    )
    eps_results[decay] = train_agent(
        ag, n_games=EPS_GAMES, n_matches=N_MATCHES,
        log_dir=f'./logs_eps{decay}', tag=f'eps{decay}'
    )

xep = np.arange(1, EPS_GAMES + 1)
fig, ax = plt.subplots(figsize=(9, 5))
for decay, r in eps_results.items():
    ax.plot(xep, rolling_mean(r['win_flags']), label=f'eps_decay={decay}')
ax.set_title('Effect of Epsilon Decay Rate on Win Rate')
ax.set_xlabel('Game')
ax.set_ylabel('Win Rate (rolling-10)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('eps_decay_exp.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eps_decay_exp.png')

In [ ]:
print("""
Variant 4 — Partial Observability  (16533318 mod 7 = 4)
========================================================

State Representation
  Full-Obs DQN : 228-dim vector  = board (11) + hand (17) + action_mask (200)
  Partial DQN  : 217-dim vector  = hand (17) + action_mask (200)   [board omitted]
  Rationale    : Withholding the board slice simulates a realistic partial-
                 observability setting where an agent cannot always see the
                 current table state (e.g., distributed or asynchronous play).

Action Handling
  Both agents use epsilon-greedy selection with hard action masking (invalid
  actions receive Q = -1e9 before argmax). This guarantees legal play without
  the environment's random-fallback correction, preserving clean credit assignment.

Reward Usage
  Rewards are sparse (match-end only). The full-obs DQN assigns terminal reward
  to the last transition per match. The LSTM-DQN propagates the terminal reward
  through complete episode sequences via truncated BPTT (seq_len=8), allowing
  earlier hidden states to be credited for the final outcome.

Memory Mechanism
  The LSTM hidden state (h, c) is reset at the start of each match and persists
  across every action within that match. At inference time this enables implicit
  tracking of board history from the sequence of action masks and hand changes.

Limitations and Failure Modes
  1. Truncated BPTT (seq_len=8) may miss long-range dependencies in lengthy matches.
  2. Terminal-only reward with no shaping makes the first ~batch_size matches nearly
     random; convergence requires more games than the full-obs baseline.
  3. The partial agent cannot distinguish between a board that was never played on
     and a board that was cleared by a Pizza event—both appear as zero-mask changes.
  4. Multi-agent non-stationarity: opponents here are fixed RandomAgents. Against
     adaptive opponents the Q-function would need continuous retraining.
  5. Episode buffer grows linearly with game length; very long games may crowd out
     short, informative episodes in the fixed-size deque.
""")

In [ ]:
os.makedirs('./saved_models', exist_ok=True)
torch.save(full_agent.policy_net.state_dict(),
           './saved_models/full_obs_dqn.pt')
torch.save(lstm_agent.policy_net.state_dict(),
           './saved_models/lstm_partial_dqn.pt')
print('Models saved to ./saved_models/')
print()
print('Output files:')
for f in ['learning_curves.png', 'stability.png',
          'hp_hidden_size.png', 'eps_decay_exp.png']:
    print(f'  {f}')